In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.formula.api as smf
import statsmodels.stats.descriptivestats as smd
import statsmodels.api as sm
import scipy.stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.append('../')
import plotting

# Load melting curve data from normal qPCR

In [ ]:
df = pd.read_csv("../data/qpcr/Primer3_meltingCurve.csv")

df

# prepare df

In [ ]:
df = df.loc[:, ~df.columns.str.startswith('X.')]
df = df.rename(columns={"X": "A: temperature"})

df.columns = df.columns.str.split(': ').str[1]

df

# average duplicates

In [ ]:
# go from wide to long format
df = df.melt(id_vars=["temperature"], var_name="sequence", value_name="value")

# average over replicates
df = df.groupby(["temperature", "sequence"]).mean().reset_index()

df

In [ ]:
colormap = {
    "no motif": "#969696",
    "3 motif": "#de2d26",
    "5 motif": "#3182bd",
}

fig = px.line(
    df,
    x="temperature",
    y="value",
    color="sequence",
    color_discrete_map=colormap,
    render_mode="svg"
)

# no motif, direct
fig.add_vline(
    x=77.0,
    line_width=2,
    # line_dash="dash",
    line_color=colormap['no motif'],
)

# 5' motif, direct
fig.add_vline(
    x=78.4,
    line_width=2,
    # line_dash="dash",
    line_color=colormap['5 motif'],
)

# 3' motif, direct
fig.add_vline(
    x=78.1,
    line_width=2,
    # line_dash="dash",
    line_color=colormap['3 motif'],
)

# 5' motif, hairpin
fig.add_vline(
    x=80.9,
    line_width=2,
    line_dash="dash",
    line_color=colormap['5 motif'],
)

# 3' motif, hairpin
fig.add_vline(
    x=81.1,
    line_width=2,
    line_dash="dash",
    line_color=colormap['3 motif'],
)



fig.update_layout(
    margin=dict(l=0, r=0, t=12, b=0),
    height=200,
    width=680,
    showlegend=False,
)
fig.update_xaxes(title="Temperature (°C)", range=[70, 90])
fig.update_yaxes(title="-dF/dT", range=[0, 10])
fig = plotting.standardize_plot(fig)
fig.show()
fig.write_image("./SI_figure_primer3_sequences/melting_curve_qpcr.svg",)